# LC 787 — Cheapest Flights Within K Stops
**Difficulty:** Medium | **Category:** Graph | **Pattern:** Bellman-Ford / Modified Dijkstra

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> The K-stop constraint prevents
standard Dijkstra (greedy settling breaks with hop limits).
Use Bellman-Ford with exactly k+1 rounds, copying dist before
each round so same-round updates don't compound.
</div>

## Official Problem Statement

There are `n` cities connected by some number of flights.
You are given an array `flights` where
`flights[i] = [fromi, toi, pricei]` indicates there is a
flight from city `fromi` to city `toi` with cost `pricei`.

Given three integers `src`, `dst`, and `k`, return the
**cheapest price** from `src` to `dst` with **at most k stops**.
If there is no such route, return `-1`.

**Constraints:**
- `1 <= n <= 100`
- `0 <= flights.length <= (n * (n - 1) / 2)`
- `flights[i].length == 3`
- `0 <= fromi, toi < n`
- `fromi != toi`
- `1 <= pricei <= 10000`
- There will not be any multiple flights between two cities
- `0 <= k < n`
- There are no cycles

## What This Is Actually Asking

You want to fly from city A to city Z as cheaply as possible,
but you can only make at most K layover stops in between.

A direct flight is 0 stops. One connection is 1 stop.
So "at most K stops" means you can take at most K+1 flights.

The twist: a cheaper total route might need more hops than
allowed. You must find the cheapest route within the hop limit,
not the globally cheapest route.

## Walk Through an Example by Hand

```
n=4, flights=[[0,1,100],[1,2,100],[2,3,100],[0,3,500]]
src=0, dst=3, k=1
```

Paths from 0 to 3:
- 0->3 direct: cost=500, stops=0  (within k=1)
- 0->1->2->3:  cost=300, stops=2  (exceeds k=1, INVALID)
- 0->1->3:     no such edge

**Bellman-Ford rounds (k+1 = 2 rounds):**

```
Init dist = [0, inf, inf, inf]

Round 1 (using copy of dist):
  Edge 0->1 (100): dist[1] = min(inf, 0+100) = 100
  Edge 0->3 (500): dist[3] = min(inf, 0+500) = 500
  Edge 1->2 (100): dist[2] = min(inf, inf+100) = inf (src inf)
  Edge 2->3 (100): dist[3] = min(500, inf+100) = 500
  dist after round 1: [0, 100, inf, 500]

Round 2 (using copy of dist from round 1):
  Edge 1->2 (100): dist[2] = min(inf, 100+100) = 200
  Edge 2->3 (100): dist[3] = min(500, inf+100) = 500 (200 not yet)
  dist after round 2: [0, 100, 200, 500]
```

Answer: dist[3] = **500**

## The Picture

**Weighted directed graph:**
```
  [0] ---100---> [1] ---100---> [2]
   |                              |
  500                            100
   |                              |
   +---------> [3] <-------------+
```

With k=1 (at most 1 stop = at most 2 flights):
- Path 0->3 uses 1 flight (0 stops) — valid, cost 500
- Path 0->1->2->3 uses 3 flights (2 stops) — BLOCKED by k=1

**Bellman-Ford dist array after each round:**
```
           node:   0     1     2     3
Init:           [  0,  inf,  inf,  inf ]
After round 1:  [  0,  100,  inf,  500 ]  <- only 1-hop paths
After round 2:  [  0,  100,  200,  500 ]  <- 2-hop paths added
                                     ^
                        answer = dist[dst=3] = 500
```

Key: we copy dist BEFORE each round so that within one round
we only extend paths by exactly ONE more edge. This enforces
the hop count correctly.

## When To Use This Pattern

- When you see **shortest path + max hops/stops constraint**,
  think Bellman-Ford (not Dijkstra — greedy breaks here).
- When you see **"at most K intermediate nodes"**, run exactly
  K+1 relaxation rounds.
- When you need to **prevent same-round chaining**, copy the
  dist array before each relaxation round.
- When the answer is about a **single destination** (not all
  nodes), Bellman-Ford is simpler than modified Dijkstra.
- When weights are **positive but hops are bounded**, Bellman-
  Ford with K+1 rounds is the clean standard approach.

## The Approach

Initialize a dist array of size n to infinity, set dist[src]=0.
Then run exactly k+1 relaxation rounds of Bellman-Ford.

Before each round, take a snapshot (copy) of the current dist.
Relax every edge using only the snapshot as the source of
costs — this ensures each round adds exactly one more hop and
prevents chaining multiple edges within the same round.

After all rounds, return dist[dst] if it is not infinity,
otherwise return -1.

In [ ]:
import heapq          # min-heap (used in Dijkstra variant)
from typing import List  # type hints for function signatures

In [ ]:
def test_harness(func):
    """Run test cases for LC 787 - Cheapest Flights K Stops."""
    tests = [
        # (n, flights, src, dst, k, expected)
        # Basic LeetCode example
        (
            4,
            [[0,1,100],[1,2,100],[2,3,100],[0,3,500]],
            0, 3, 1, 500
        ),
        # Enough stops to take cheaper path
        (
            4,
            [[0,1,100],[1,2,100],[2,3,100],[0,3,500]],
            0, 3, 2, 300
        ),
        # No path exists
        (
            3,
            [[0,1,100],[0,2,500]],
            0, 2, 0, 500
        ),
        # Direct flight only, k=0
        (
            2,
            [[0,1,200]],
            0, 1, 0, 200
        ),
        # Unreachable with k=0 (need 1 stop)
        (
            3,
            [[0,1,100],[1,2,100]],
            0, 2, 0, -1
        ),
    ]

    passed = 0
    for i, (n, flights, src, dst, k, expected) in enumerate(tests):
        result = func(n, flights, src, dst, k)
        status = "PASSED" if result == expected else "FAILED"
        if status == "PASSED":
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"got={result}, expected={expected}"
        )

    print(f"\nResult: {passed}/{len(tests)} tests passed")

In [ ]:
def find_cheapest_price(
    n: int,
    flights: List[List[int]],
    src: int,
    dst: int,
    k: int
) -> int:
    """
    LC 787 - Cheapest Flights Within K Stops.

    Restatement:
        Find the cheapest path from src to dst using at most
        k intermediate stops (k+1 total flights). Return -1
        if no valid path exists within the hop limit.

    Approach:
        Bellman-Ford with k+1 rounds. Before each round, copy
        dist so relaxations use previous-round costs only.
        This enforces exactly one new hop per round.

    Time:  O(k * E) — k+1 rounds, each processes all edges
    Space: O(n) — dist array + temp copy per round
    """
    pass


# Direct test prints — expected values shown in comments
print(find_cheapest_price(
    4, [[0,1,100],[1,2,100],[2,3,100],[0,3,500]], 0, 3, 1
))
# Expected: 500

print(find_cheapest_price(
    4, [[0,1,100],[1,2,100],[2,3,100],[0,3,500]], 0, 3, 2
))
# Expected: 300

print(find_cheapest_price(
    3, [[0,1,100],[1,2,100]], 0, 2, 0
))
# Expected: -1

print(find_cheapest_price(
    2, [[0,1,200]], 0, 1, 0
))
# Expected: 200

In [ ]:
# Uncomment and run when solution is ready
# test_harness(find_cheapest_price)

## Complexity

| Approach                  | Time        | Space  |
|---------------------------|-------------|--------|
| Brute Force DFS           | O(n^k)      | O(k)   |
| Bellman-Ford (k+1 rounds) | O(k * E)    | O(n)   |
| Modified Dijkstra         | O(k*E log n)| O(n*k) |

## Real World Connection

At Citi, AWS data pipelines often route events through a
bounded number of intermediate services (e.g., Kinesis ->
Lambda -> S3 -> Glue). Adding more hops increases latency
and cost — this is the K-stops constraint in production.

When designing VPC traffic routing across AWS regions, a
network engineer may limit traffic to at most 2 transit
gateway hops for latency SLA compliance, then minimize cost
within that constraint — exactly LC 787's problem.

In distributed data engineering, "cheapest path within K
hops" maps to: find the lowest-cost ETL pipeline topology
that still meets the max-hop governance policy. Bellman-Ford
rounds naturally model the step-by-step pipeline propagation.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra